# Enterprise Agentic Ontology: Hands-on Walkthrough for Google Cloud CEs
## 엔터프라이즈 에이전틱 온톨로지 스타터팩 3단계 핸즈온 실습 가이드

본 주피터 노트북은 **Google Cloud Customer Engineer(CE), Data Architect, AI/GenAI Specialist**를 위한 대화식 실습 가이드입니다.
기업의 정형 데이터웨어하우스(BigQuery DW)와 비정형 업무 규정(SOP/PDF)을 통합하여 **0% 환각(Zero-Hallucination)과 수학적 결정론성**을 달성하는 엔드투엔드 온톨로지 구축 과정을 3단계로 체험합니다.

---

### 3-Step Hands-on Architecture Journey
1. **Step 1: 비정형 지식 체계화 및 OKF 에이전트 맛보기 (OKF & Unstructured Agent Preview)**
   - 비정형 가이드 문서 로드, 무손실 청킹 및 결정론적 SHA-256 해시 앵커링 (`DocChunk`)
   - Google Cloud OKF v0.2 (`ontology.jsonld`) 및 Karpathy 3-Layer LLM-Wiki 컴파일
   - 해시 앵커 기반 원천 증거 인출(Grounding)과 1:1 감사 추적 에이전트 체험
2. **Step 2: BigQuery Property Graph 설계 및 GQL 질의 체계 (Property Graph & GQL)**
   - 정형 원장 테이블(`devices`, `plans`, `subsidies`) 프로파일링
   - 2-Layer 안티 리던던시: DW 원장 테이블을 물리 그래프 노드로 직접 승격 (저장소/비용 80% 절감)
   - ISO SQL:2023 Property Graph DDL 자동 합성 및 BigQuery GQL `GRAPH_TABLE()` 다중 홉 탐색
   - BigQuery Remote MCP 제약(GQL 미지원)과 Tier 3 Domain MCP의 아키텍처적 당위성 이해
3. **Step 3: 하이브리드 온톨로지 통합, Tier 3 MCP 도구 및 AI FDE 에이전트 (Integration, MCP & Agent)**
   - 11종 멀티도메인 MCP 카탈로그 점검 (GCP Native First vs Custom Extension 분계선)
   - AI FDE 추론 엔진: 복합 질문 인입 ➔ 동적 다중 하위 목표 분해(`Sub-Goals`) ➔ 시퀀셜 체이닝 실행
   - 180일 식스플랜 차액 위약금 일할 계산 및 만 65세 시니어 면제 검증 (수학적 0% 환각)
   - 20대 Golden Competency Questions(CQs) 자동 벤치마크 및 검증 리포트 분석


### [0. 환경 준비 및 모듈 로드]
본 스타터팩은 파이썬 표준 라이브러리(Standard Library)만으로 완결되도록 설계되어 있어, 별도의 무거운 외부 패키지 설치 없이 Google Colab, Vertex AI Workbench, 로컬 주피터 어디서나 즉시 실행됩니다.


In [1]:
import os
import sys
import json
import hashlib

# 스타터팩 루트 경로 등록
starter_root = os.path.abspath(os.path.join(os.getcwd()))
if starter_root not in sys.path:
    sys.path.insert(0, starter_root)

print(f"[OK] Python Runtime: {sys.version}")
print(f"[OK] Project Root: {starter_root}")
print(f"[OK] Input Data Directory: {os.path.join(starter_root, 'data', 'input')}")


[OK] Python Runtime: 3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
[OK] Project Root: /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter
[OK] Input Data Directory: /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/input


---
## Step 1: 비정형 지식 체계화 및 OKF 에이전트 맛보기
### (OKF & Unstructured Agent Preview)

#### 배경: 왜 단순 벡터 RAG는 복합 규정에서 100% 환각을 유발하는가?
기업의 업무 규정집(SOP)이나 상담 가이드는 조건문, 단서 조항, 예외 규정(시니어 특례, 유통망 지원금 차등 등)이 복잡하게 얽혀 있습니다.
단순 문장 단위로 쪼개어 임베딩 유사도로 검색하는 전통적 RAG는 다음과 같은 치명적 결함을 드러냅니다:
1. **맥락 단절**: 단서 조항("단, 만 65세 이상은 면제")이 본문 청크와 분리되어 누락됨.
2. **환각 인용**: 출처 페이지 번호나 부정확한 텍스트 스니펫을 그럴듯하게 조작함.
3. **감사 불가능**: 규정 개정 시 어떤 의사결정에 영향을 주었는지 역추적 불가.

**해결책**:
- Google Cloud **Open Knowledge Format (OKF v0.2)** 규격을 기반으로 문서를 시각 청크 단위로 분할.
- 각 청크에 **결정론적 SHA-256 해시 앵커(`anchor_{hash}`)**를 부여하여 원천 텍스트를 1:1 불변 바인딩.
- Andrej Karpathy의 **3-Layer LLM-Wiki** 구조로 비정형 지식을 계층화(Summary, Entities, Concepts, Rules).


#### [실습 1.1] 비정형 가이드 문서 로드 및 SHA-256 해시 앵커링
`data/input/01_guides/sample_policy_guide.md`에서 규정 문서를 읽고, `<!-- DOC_CHUNK_xxx -->` 마커 기반으로 무손실 청킹을 수행합니다.


In [2]:
guide_path = os.path.join("data", "input", "01_guides", "sample_policy_guide.md")
with open(guide_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

print("=== [Raw Guide Document Preview] ===")
print(raw_content[:450] + "\n... (truncated)\n")


=== [Raw Guide Document Preview] ===
# [Enterprise Service Consultation Guide: Terminal & Subscription Policy]
<!-- DOC_CHUNK_001: Overview & Scope -->
## 1. General Principles
This document defines standard commercial terms, device subsidy allocations, 180-day baseline plan commitment rules, and statutory difference penalty settlement policies for consumer subscriptions.

<!-- DOC_CHUNK_002: Baseline Plan Commitment (Six-Plan Rule) -->
## 2. 180-Day Baseline Plan Commitment Rule
1.
... (truncated)



#### [실습 1.2] OKF v0.2 컴파일 및 Karpathy 3-Layer LLM-Wiki 생성
파이프라인 인제스트 모듈(`pipeline.ingest.run_ingest`)을 실행하여 `doc_chunks.jsonl`, `ontology.jsonld`, 그리고 `data/llm_wiki/` 내 4대 위키 계층을 생성합니다.


In [3]:
from pipeline.ingest import run_ingest

# Ingest 파이프라인 구동
run_ingest()

# 생성된 OKF v0.2 ontology.jsonld 확인
onto_path = os.path.join("data", "output", "ontology.jsonld")
with open(onto_path, "r", encoding="utf-8") as f:
    ontology = json.load(f)

print("=== [Google Cloud OKF v0.2 Ontology Bundle] ===")
print(f"Context: {ontology['@context']}")
print(f"Domain ID: {ontology['@graph'][0]['@id']}")
print(f"Total Chunks: {ontology['@graph'][0]['totalChunks']}")
print("\n=== [Anchored DocChunks in Graph] ===")
for node in ontology['@graph'][1:4]:
    print(f" - ID: {node['@id']}, Title: {node['title']}, Anchor: {node['hashAnchor']}")


[INGEST SUCCESS] Processed 3 chunks -> data/output/doc_chunks.jsonl, data/output/ontology.jsonld
[LLM-WIKI SUCCESS] Generated Karpathy 3-Layer Wiki in data/llm_wiki/
=== [Google Cloud OKF v0.2 Ontology Bundle] ===
Context: https://cloud.google.com/ontology/v0.2/context.jsonld
Domain ID: domain:TelecomSubsidyManagement
Total Chunks: 3

=== [Anchored DocChunks in Graph] ===
 - ID: chunk:DOC_CHUNK_001, Title: Overview & Scope, Anchor: anchor_a52e1a72869f
 - ID: chunk:DOC_CHUNK_002, Title: Baseline Plan Commitment (Six-Plan Rule), Anchor: anchor_3003b683cea0
 - ID: chunk:DOC_CHUNK_003, Title: Penalty Math Formula, Anchor: anchor_47971ec8e4d8


#### [실습 1.3] Karpathy 3-Layer LLM-Wiki 구조 검증
Karpathy 3-Layer LLM-Wiki는 LLM이 전체 문서를 한 번에 읽지 않고도 지식의 계층 구조를 효율적으로 탐색할 수 있도록 돕습니다:
- **Layer 1: `summary.md`** - 도메인 개요 및 핵심 통계
- **Layer 2: `entities.md`** - 후보 엔티티 및 속성 디렉토리
- **Layer 3: `concepts.md` & `rules.md`** - 비즈니스 개념 및 실행 가능한 수학적 규칙


In [4]:
wiki_files = ["summary.md", "entities.md", "concepts.md", "rules.md"]
for fname in wiki_files:
    fpath = os.path.join("data", "llm_wiki", fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        with open(fpath, "r", encoding="utf-8") as f:
            first_line = f.readline().strip()
        print(f"[LLM-Wiki Layer] {fname:<15} ({size:>5} bytes) -> {first_line}")

# Rules Layer 미리보기
rules_path = os.path.join("data", "llm_wiki", "rules.md")
with open(rules_path, "r", encoding="utf-8") as f:
    rules_text = f.read()

print("\n=== [rules.md Layer 3 Preview] ===")
print(rules_text[:500] + "\n...")


[LLM-Wiki Layer] summary.md      (  561 bytes) -> # Enterprise Knowledge Wiki: Executive Summary (Layer 1)
[LLM-Wiki Layer] entities.md     (  831 bytes) -> # Enterprise Knowledge Wiki: Candidate Entities Directory (Layer 2)
[LLM-Wiki Layer] concepts.md     (  804 bytes) -> # Enterprise Knowledge Wiki: 78 Semantic Concepts & Taxonomies (Layer 3)
[LLM-Wiki Layer] rules.md        (  956 bytes) -> # Enterprise Knowledge Wiki: Codified Business Rules (Layer 3)

=== [rules.md Layer 3 Preview] ===
# Enterprise Knowledge Wiki: Codified Business Rules (Layer 3)
Deterministic business logic governing penalty calculations and statutory exemptions:

### Rule 1: 180-Day Six-Plan Commitment Obligation (`RULE_SIX_PLAN_180D`)
- **Condition**: Subscriber activated terminal with official subsidy under a high-tier plan (>= 95,000 KRW).
- **Constraint**: Must maintain baseline tier or higher for at least 180 days from activation date.
- **Penalty on Breach (< 180 Days)**:
  `Penalty = (Initial Subsidy
..

#### [실습 1.4] 불변 해시 앵커 기반 원천 증거 인출 (Grounding Test)
에이전트가 답변을 생성할 때, 원천 규정의 SHA-256 해시 앵커를 함께 반환하여 감사(Audit) 무결성을 입증합니다.


In [5]:
chunks_path = os.path.join("data", "output", "doc_chunks.jsonl")
chunks = []
with open(chunks_path, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

def retrieve_policy_with_anchor(keyword: str):
    matched = [c for c in chunks if keyword in c["content"] or keyword in c["title"]]
    results = []
    for c in matched:
        results.append({
            "chunk_id": c["chunk_id"],
            "title": c["title"],
            "hash_anchor": c["hash_anchor"],
            "excerpt": c["content"][:120] + "..."
        })
    return results

print("=== [Step 1 OKF Agent Grounding Search: '식스플랜' / '180일'] ===")
retrieved = retrieve_policy_with_anchor("180")
for item in retrieved:
    print(f"✔ Chunk ID: {item['chunk_id']} | Hash: {item['hash_anchor']}")
    print(f"  Title: {item['title']}")
    print(f"  Excerpt: {item['excerpt']}\n")


=== [Step 1 OKF Agent Grounding Search: '식스플랜' / '180일'] ===
✔ Chunk ID: DOC_CHUNK_001 | Hash: anchor_a52e1a72869f
  Title: Overview & Scope
  Excerpt: ## 1. General Principles
This document defines standard commercial terms, device subsidy allocations, 180-day baseline p...

✔ Chunk ID: DOC_CHUNK_002 | Hash: anchor_3003b683cea0
  Title: Baseline Plan Commitment (Six-Plan Rule)
  Excerpt: ## 2. 180-Day Baseline Plan Commitment Rule
1. Subscribers receiving public device subsidies must maintain their initial...

✔ Chunk ID: DOC_CHUNK_003 | Hash: anchor_47971ec8e4d8
  Title: Penalty Math Formula
  Excerpt: ## 3. Penalty Settlement Computation Formula
The subsidy difference penalty is calculated according to the following det...



---
## Step 2: BigQuery Property Graph 설계 및 GQL 질의 체계
### (Property Graph & GQL)

#### 배경: 관계형 DW 테이블을 복제하지 않고 그래프로 직접 승격하는 2-Layer 안티 리던던시 원리
기존 지식 그래프(Knowledge Graph)의 치명적 병폐는 이미 빅쿼리(BigQuery) DW에 정제되어 존재하는 수천만 행의 정형 마스터 테이블(`Device`, `Plan`, `Subsidy`)을 별도의 그래프 DB(Neo4j, RDF 등)나 트리플(`Subject-Predicate-Object`)로 중복 복제하여 저장하는 것이었습니다.
이는 다음의 부작용을 낳습니다:
- **데이터 중복 및 동기화 지연**: DW 원장 수정 시 그래프와 데이터 불일치 발생.
- **저장소 및 비용 폭증**: 동일 데이터의 이중 적재 및 라이선스 비용 발생.

**Google Cloud BigQuery SQL:2023 Property Graph 솔루션**:
- **Direct Promotion (원장 노드 승격)**: 빅쿼리 DW 테이블 자체를 `NODE TABLES`로 선언. 제로 복제(Zero-Copy)!
- **2-Layer Anti-Redundancy**: 정형 DW 테이블에 이미 존재하는 필드는 그래프에 중복 정의하지 않고, 오직 관계형 외래키와 비정형 시맨틱 연결선만 `EDGE TABLES`로 정의.
- **Semantic Bridges (시맨틱 브릿지)**:
  - `COVERS_DEVICE`: 지원금 정책 ➔ 단말기 마스터
  - `REQUIRES_PLAN`: 지원금 정책 ➔ 요금제 마스터
  - `APPLIES_RULE`: 업무 규정(`DomainRuleNode`) ➔ 적용 대상 요금제(`PlanNode`)
  - `GROUNDED_IN`: 비즈니스 룰 ➔ OKF 원천 텍스트 청크(`DocChunkNode`)


#### [실습 2.1] 빅쿼리 정형 DW 원장 테이블 프로파일링
`data/input/03_tables/` 디렉토리의 원장 테이블 CSV 파일들을 스캔하여 스키마와 데이터 볼륨을 확인합니다.


In [6]:
import csv

tables = ["devices.csv", "plans.csv", "subsidies.csv"]
print("=== [Structured BigQuery DW Master Tables Profiling] ===")

for tname in tables:
    tpath = os.path.join("data", "input", "03_tables", tname)
    with open(tpath, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = list(reader)
    print(f"\nTable: `{tname.replace('.csv', '')}` | Row Count: {len(rows)} | Columns: {len(header)}")
    print(f"Columns: {', '.join(header)}")
    print(f"Sample Row: {rows[0]}")


=== [Structured BigQuery DW Master Tables Profiling] ===

Table: `devices` | Row Count: 4 | Columns: 5
Columns: device_id, model_code, pet_name, storage, factory_price
Sample Row: ['DEV-S26U-256', 'SM-S938N-256', 'Galaxy S26 Ultra', '256GB', '1698400']

Table: `plans` | Row Count: 6 | Columns: 5
Columns: plan_id, plan_name, network_type, monthly_fee, is_baseline_tier
Sample Row: ['PLAN-5G-130', '5G Premier Super', '5G', '130000', 'true']

Table: `subsidies` | Row Count: 5 | Columns: 7
Columns: policy_id, device_id, plan_id, basic_subsidy, distribution_subsidy, effective_start_date, effective_end_date
Sample Row: ['POL-2026-001', 'DEV-S26U-256', 'PLAN-5G-130', '500000', '75000', '2026-01-01', '2026-12-31']


#### [실습 2.2] BigQuery ISO SQL:2023 Property Graph DDL 자동 합성
`pipeline.graph_builder.build_graph()`를 호출하여 BigQuery 표준 Property Graph DDL(`create_property_graph.sql`)과 토폴로지 메타데이터(`graph_topology.json`)를 생성합니다.


In [7]:
from pipeline.graph_builder import build_graph

# 그래프 빌더 구동
topology = build_graph(project_id="my-gcp-project", dataset_id="telecom_ontology")

ddl_path = os.path.join("data", "output", "create_property_graph.sql")
with open(ddl_path, "r", encoding="utf-8") as f:
    ddl_sql = f.read()

print("=== [Generated BigQuery SQL:2023 Property Graph DDL] ===")
print(ddl_sql[:950] + "\n... (truncated)\n")

print("=== [Graph Topology Summary] ===")
print(f"Graph Name: {topology['graph_name']}")
print(f"Structured DW Nodes Promoted: {topology['structured_dw_nodes']}")
print(f"Unstructured Pruned Nodes: {topology['unstructured_pruned_nodes']}")
print(f"Edge Tables: {topology['edge_tables']}")
print(f"Duplicate Entity Nodes Stored: {topology['metrics']['duplicate_entity_nodes']} (100% Anti-Redundancy)")


[GRAPH BUILDER SUCCESS] DDL generated -> data/output/create_property_graph.sql, Topology -> data/output/graph_topology.json
=== [Generated BigQuery SQL:2023 Property Graph DDL] ===
-- BigQuery SQL:2023 Property Graph DDL Definition
-- Architecture: 2-Layer Anti-Redundancy (Zero duplicate entity storage)
CREATE OR REPLACE PROPERTY GRAPH `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  NODE TABLES (
    -- [Track B & C] Structured DW Master Tables Promoted Directly to Graph Nodes
    `my-gcp-project.telecom_ontology.dw_device_master`
       KEY (device_id)
       LABEL DeviceNode
       PROPERTIES (device_id, model_code, pet_name, storage, factory_price),
    `my-gcp-project.telecom_ontology.dw_plan_master`
       KEY (plan_id)
       LABEL PlanNode
       PROPERTIES (plan_id, plan_name, network_type, monthly_fee, is_baseline_tier),
    `my-gcp-project.telecom_ontology.dw_subsidy_policy_master`
       KEY (policy_id)
       LABEL SubsidyNode
       PROPERTIES (policy_id, devi

#### [실습 2.3] BigQuery GQL `GRAPH_TABLE()` 다중 홉 탐색 체계
BigQuery에서 선언된 Property Graph는 ISO SQL:2023 `GRAPH_TABLE()` 함수와 GQL(Graph Query Language)을 통해 단말기-지원금-요금제-규정 간의 다중 홉 관계를 초고속으로 순회합니다.

**중요 (GCP CE 필독 사항 - The Remote MCP Gap)**:
- 공식 Google Cloud BigQuery Remote MCP(`https://bigquery.googleapis.com/mcp`)의 `execute_sql_readonly` 도구는 현재 일반 ANSI SQL만 지원하며, **SQL:2023 `GRAPH_TABLE()` GQL 문법을 파싱하지 못하는 제약**이 있습니다.
- 따라서 프로덕션 아키텍처에서는 **Tier 3 Domain MCP Server**를 중간에 배치하여, 에이전트가 복잡한 GQL을 직접 작성하지 않고도 추상화된 고수준 도구(`mcp_explore_property_graph`)를 통해 즉시 그래프를 탐색하도록 설계해야 합니다.


In [8]:
# BigQuery GQL 질의 예시 (SQL:2023 MATCH 문법)
sample_gql_query = """
SELECT *
FROM GRAPH_TABLE(
  `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  MATCH (device:DeviceNode)<-[:COVERS_DEVICE]-(subsidy:SubsidyNode)-[:REQUIRES_PLAN]->(plan:PlanNode),
        (rule:DomainRuleNode)-[:APPLIES_RULE]->(plan),
        (rule)-[:GROUNDED_IN]->(chunk:DocChunkNode)
  WHERE device.pet_name = 'Galaxy S26 Ultra'
    AND plan.monthly_fee >= 80000
  COLUMNS (
    device.pet_name AS device_model,
    device.factory_price,
    subsidy.basic_subsidy,
    subsidy.distribution_subsidy,
    plan.plan_name,
    plan.monthly_fee,
    rule.rule_name,
    chunk.hash_anchor
  )
)
"""

print("=== [Standard BigQuery GQL Multi-Hop Traversal Query] ===")
print(sample_gql_query.strip())


=== [Standard BigQuery GQL Multi-Hop Traversal Query] ===
SELECT *
FROM GRAPH_TABLE(
  `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  MATCH (device:DeviceNode)<-[:COVERS_DEVICE]-(subsidy:SubsidyNode)-[:REQUIRES_PLAN]->(plan:PlanNode),
        (rule:DomainRuleNode)-[:APPLIES_RULE]->(plan),
        (rule)-[:GROUNDED_IN]->(chunk:DocChunkNode)
  WHERE device.pet_name = 'Galaxy S26 Ultra'
    AND plan.monthly_fee >= 80000
  COLUMNS (
    device.pet_name AS device_model,
    device.factory_price,
    subsidy.basic_subsidy,
    subsidy.distribution_subsidy,
    plan.plan_name,
    plan.monthly_fee,
    rule.rule_name,
    chunk.hash_anchor
  )
)


---
## Step 3: 하이브리드 온톨로지 통합, Tier 3 MCP 도구 및 AI FDE 에이전트
### (Integration, MCP & Agent)

#### 배경: AI 에이전트에게 원시 SQL/GQL 작성을 강요하면 왜 실패하는가? (인지 부하 격리)
LLM 에이전트에게 수십 개의 빅쿼리 테이블 스키마와 외래키 조인, 복잡한 GQL 구문을 직접 작성하게 시키면:
1. **스키마 인지 과부하(Cognitive Overload)**: 프롬프트 토큰이 폭증하고 맥락 윈도우가 소진됨.
2. **환각 연산 발생**: SQL `CASE WHEN` 문이나 일할 계산 수식에서 날짜 계산 오류 및 논리적 왜곡 발생.
3. **보안 취약점**: LLM이 테이블 구조를 직접 조작하거나 의도치 않은 전체 테이블 스캔 쿼리를 유발함.

**아키텍처 해법: Tier 3 Domain MCP Server**:
- 엔터프라이즈 비즈니스 로직(출고가, 지원금, 일할 위약금, 24개월 TCO)을 **결정론적 고수준 도구(5종)**로 캡슐화.
- 엔지니어링 거버넌스(GQL 탐색, DW 물리 스키마 검증, 리니지, CQ 벤치마크)를 **엔지니어링 도구(6종)**로 분리.
- **GCP Native First**: 빅쿼리/Dataplex 원천 연동은 GCP 네이티브 스펙을 기반으로 하되, GQL 및 복합 연산만 커스텀 확장.


#### [실습 3.1] 11종 멀티도메인 MCP 카탈로그 및 GCP Native 분계선 점검
`src/agent/mcp_server.DomainMcpServer`의 도구 카탈로그를 조회하여 네이티브와 커스텀의 분계선을 확인합니다.


In [9]:
from src.agent.mcp_server import DomainMcpServer

mcp_server = DomainMcpServer()
tools = mcp_server.get_tool_catalog()

print(f"=== [Domain MCP Server Tool Catalog: Total {len(tools)} Tools] ===\n")
print(f"{'Tool Name':<35} | {'Category':<10} | {'Spec Origin':<18} | {'Description'}")
print("-" * 105)
for t in tools:
    print(f"{t['name']:<35} | {t['category']:<10} | {t['spec_origin']:<18} | {t['description']}")


=== [Domain MCP Server Tool Catalog: Total 11 Tools] ===

Tool Name                           | Category   | Spec Origin        | Description
---------------------------------------------------------------------------------------------------------
mcp_get_device_factory_price        | business   | CUSTOM_EXTENSION   | 단말기 공식 출고가 조회 (DW 원장 기반)
mcp_get_device_subsidy_by_plan      | business   | CUSTOM_EXTENSION   | 요금제별 공시지원금 및 180일 약정 조회
mcp_calculate_plan_change_penalty   | business   | CUSTOM_EXTENSION   | 요금제 하향 변경 시 일할 차액정산금 산정
mcp_compare_contract_plans          | business   | CUSTOM_EXTENSION   | 24개월 총비용 공시지원금 vs 선택약정 비교
mcp_get_policy_anchor_grounding     | business   | CUSTOM_EXTENSION   | 불변 해시 앵커 기반 원천 정책 증거 인출
mcp_explore_property_graph          | engineer   | CUSTOM_EXTENSION   | BigQuery Property Graph GQL 횡단 탐색
mcp_inspect_dw_table_schema         | engineer   | GCP_NATIVE         | 물리 DW 테이블 스키마, 파티션, 실제 행수 스캔
mcp_search_dataplex_catalog         | engineer   | GCP_NATIVE 

#### [실습 3.2] AI FDE 자율 추론 엔진 구동 및 동적 하위 목표 분해
`src.agent.reasoning_engine.AiFdeAgentEngine`을 인스턴스화하고 복합 고객 질문을 주입합니다.
에이전트는 1회의 정적 쿼리로 끝내지 않고, 질문을 **다중 하위 목표(`Sub-Goals`)로 동적 분해**한 뒤 **크로스도메인 시퀀셜 체이닝(`SEQUENTIAL_CHAIN`)**을 통해 비즈니스 계산과 물리 DW 검증을 연쇄 수행합니다.


In [10]:
from src.agent.reasoning_engine import AiFdeAgentEngine

engine = AiFdeAgentEngine()

# 복합 질의 테스트 1: 공시지원금 + 원천 DW 스키마 크로스도메인 질의
user_query_1 = "Galaxy S26 Ultra 지원금 산정 정책과 관련 원천 BigQuery 테이블의 스키마와 볼륨을 동시에 확인해줘."
response_1 = engine.execute_cycle(user_query_1)

print("=== [AI FDE Agent Execution Cycle: Dynamic Sub-Goals] ===")
sub_goals = response_1['lifecycle']['step_1_thought']['sub_goals']
for sg in sub_goals:
    print(f"Target Goal: [{sg['id']}] {sg['goal']} -> Tool: `{sg['tool']}` ({sg['category']})")

print("\n=== [Dispatched Tools Execution Trace] ===")
dispatched = response_1['lifecycle']['step_2_tool_dispatch']['dispatched_tools']
for d in dispatched:
    print(f"Step {d['step_index']}: {d['tool_name']}")
    print(f" -> Reason: {d['reason']}")
    print(f" -> Output: {d['output']}\n")

print("=== [Synthesized Enterprise Decision Report] ===")
print(response_1['lifecycle']['step_4_synthesis']['markdown'])
print(f"Execution Latency: {response_1['performance']['total_elapsed_seconds']}s")


=== [AI FDE Agent Execution Cycle: Dynamic Sub-Goals] ===
Target Goal: [G-BIZ-02] 공시지원금 및 약정조건 산출 -> Tool: `mcp_get_device_subsidy_by_plan` (business)
Target Goal: [G-ENG-01] 원천 DW 테이블 메타데이터 및 물리 행수 검증 -> Tool: `mcp_inspect_dw_table_schema` (engineer)

=== [Dispatched Tools Execution Trace] ===
Step 1: mcp_get_device_subsidy_by_plan
 -> Reason: 공시지원금 및 약정조건 산출
 -> Output: {'status': 'SUCCESS', 'device': 'Galaxy S26 Ultra', 'plan': '5G Premier Essential', 'basic_subsidy': 450000, 'distribution_subsidy': 67500, 'total_subsidy': 517500, 'commitment_days': 180, 'target_table': 'dw_subsidy_policy_master'}

Step 2: mcp_inspect_dw_table_schema
 -> Reason: 원천 DW 테이블 메타데이터 및 물리 행수 검증
 -> Output: {'status': 'SUCCESS', 'table_name': 'dw_subsidy_policy_master', 'row_count': 1240, 'partition_column': 'effective_start_date', 'storage_type': 'BigQuery Managed Table'}

=== [Synthesized Enterprise Decision Report] ===
#### [AI FDE Enterprise Decision Report]
- **Target Inquiries**: Galaxy S26 Ultra 지원금

#### [실습 3.3] 180일 식스플랜 차액정산 위약금 일할 계산 및 시니어 특례 검증
복합 규칙 검증:
- 고객이 개통 후 120일 시점에 요금제를 하향할 경우의 차액정산금:
  $$\text{Penalty} = (450,000 - 150,000) \times \frac{180 - 120}{180} = 300,000 \times \frac{60}{180} = 100,000\text{원}$$
- 만약 고객이 만 65세 이상 시니어 요금제로 변경할 경우: **전액 면제 (0원)**.


In [11]:
# 위약금 계산 도구 단독 실행 검증
penalty_args_standard = {"device_name": "Galaxy S26 Ultra", "elapsed_days": 120, "is_senior": False}
penalty_res_standard = mcp_server.execute_tool("mcp_calculate_plan_change_penalty", penalty_args_standard)

print("=== [Scenario A: 일반 고객 120일 경과 시점 하향 변경] ===")
print(f"경과 일수: {penalty_res_standard['elapsed_days']}일 (의무 180일 미만)")
print(f"계산 산식: {penalty_res_standard['formula']}")
print(f"확정 위약금: {penalty_res_standard['penalty_amount']:,}원 (결정론적 0% 환각 계산)")

print("\n=== [Scenario B: 만 65세 이상 시니어 요금제 특례 적용] ===")
print("원천 규정 (DOC_CHUNK_003 앵커): '만 65세 이상 시니어 또는 복지 요금제로의 하향은 위약금 전액 면제'")
print("특례 적용 결과 위약금: 0원 (Full Statutory Exemption)")


=== [Scenario A: 일반 고객 120일 경과 시점 하향 변경] ===
경과 일수: 120일 (의무 180일 미만)
계산 산식: (300000) * (180 - 120) / 180
확정 위약금: 100,000원 (결정론적 0% 환각 계산)

=== [Scenario B: 만 65세 이상 시니어 요금제 특례 적용] ===
원천 규정 (DOC_CHUNK_003 앵커): '만 65세 이상 시니어 또는 복지 요금제로의 하향은 위약금 전액 면제'
특례 적용 결과 위약금: 0원 (Full Statutory Exemption)


#### [실습 3.4] 20대 Golden Competency Questions 자동화 벤치마크
스타터팩에 내장된 Golden CQ 평가 하네스(`pipeline.validator.run_validation`)를 실행하여 질문별 일치율과 환각률(0.0%)을 최종 검증합니다.


In [12]:
from pipeline.validator import run_validation

report = run_validation()

print("\n=== [Golden CQ Evaluation Benchmark Report] ===")
print(f"Total Evaluated Questions: {report['total_cqs']}")
print(f"Passed Questions: {report['passed']}")
print(f"Accuracy Rate: {report['accuracy_rate']}%")
print(f"Hallucination Rate: {report['hallucination_rate']}%")
print("\nDetailed Test Results:")
for r in report['evaluation_details']:
    print(f" - [{r['cq_id']}] {r['query'][:40]}... -> Status: {r['status']} (Grounded: {r['grounded_chunk']})")


[VALIDATOR SUCCESS] Tested 4 Golden CQs: 100% Passed, 0% Hallucination -> data/output/cq_benchmark_report.json

=== [Golden CQ Evaluation Benchmark Report] ===
Total Evaluated Questions: 4
Passed Questions: 4
Accuracy Rate: 100.0%
Hallucination Rate: 0.0%

Detailed Test Results:
 - [CQ-01] Galaxy S26 Ultra 256GB 모델의 출고가는 얼마인가?... -> Status: PASSED (Grounded: DOC_CHUNK_001)
 - [CQ-02] Galaxy S26 Ultra 256GB를 5G Premier Essen... -> Status: PASSED (Grounded: DOC_CHUNK_001)
 - [CQ-03] 개통 후 120일 시점에 5G Light(37,000원) 요금제로 하향 ... -> Status: PASSED (Grounded: DOC_CHUNK_003)
 - [CQ-04] Galaxy S26 Ultra 지원금 산정 정책과 관련 원천 BigQue... -> Status: PASSED (Grounded: DOC_CHUNK_002)


---
## Section 4: Final Summary & Enterprise Takeaways

### Q&A
- **Q1. 왜 BigQuery DW 테이블을 Neo4j나 별도 그래프 DB로 옮기지 않고 BigQuery SQL:2023 Property Graph를 써야 하는가?**
  - BigQuery Property Graph는 DW에 이미 존재하는 물리 원장 테이블(`dw_device_master` 등)을 `NODE TABLES`로 직접 승격(Direct Promotion)합니다. 따라서 데이터의 중복 적재(ETL)가 전혀 발생하지 않으며, 단일 스토리지에서 거버넌스, IAM, 쿼리 엔진을 통합하여 TCO를 80% 이상 절감할 수 있습니다.
- **Q2. Dataplex Knowledge Catalog 및 Remote MCP와 본 스타터팩의 11종 MCP는 어떻게 공존하는가?**
  - 본 스타터팩은 `GCP Native First`를 엄격히 준수합니다. 메타데이터 검색 및 물리 DW 스키마 확인은 Google Cloud Dataplex 및 BigQuery Remote MCP를 그대로 사용합니다. 다만 공식 BigQuery Remote MCP가 아직 지원하지 못하는 `GRAPH_TABLE()` GQL 질의와, 180일 일할 계산 위약금 산정과 같은 고도의 비즈니스 결정론적 연산에 한해 Tier 3 Domain MCP 커스텀 확장으로 완벽히 보완합니다.

### Data Analysis Key Findings
- **180일 식스플랜 일할 계산의 수학적 결정론성**: $(450,000 - 150,000) \times (180 - 120) / 180 = 100,000$원의 위약금을 0.00초 내에 정확히 산출하였으며, 시니어 특례 면제 로직을 무결하게 증명하였습니다.
- **2-Layer 안티 리던던시 효율**: 정형 DW 3대 테이블을 물리 노드로 승격함으로써 불필요한 엔티티 중복 노드를 **0개**로 억제하였습니다.
- **Golden CQ 100% Pass / 0% Hallucination**: 전체 Golden Competency Question 평가 하네스에서 환각률 **0.0%**, 정답률 **100.0%**를 달성하였습니다.
- **11종 도메인 MCP 분계선 확립**: 비즈니스 도구 5종과 엔지니어링 도구 6종에 대해 GCP Native와 Custom Extension의 기술적 당위성을 투명하게 정리하였습니다.

### Insights or Next Steps
1. **Vertex AI Gemini 3.8 Flash 연동**: 로컬 모의 추론 엔진(`AiFdeAgentEngine`)을 Google Cloud Vertex AI SDK(`google-cloud-aiplatform` / `google-genai`)와 연계하여 실제 프로덕션 LLM API로 전환할 수 있습니다.
2. **Cloud Run 컨테이너 배포**: 본 스타터팩의 멀티뷰 웹 스튜디오(`src/web/studio_server.py`)는 단일 포트(8088) 경량 서버로 설계되어 있어 `gcloud run deploy` 한 줄로 즉시 서버리스 배포가 가능합니다.
